# Phase 34 — v1.4 Latency-Granularity + Alias-Deduplication Repair
## Specification Reviewer Notebook

**Classification:** `READ_ONLY_PHASE34_LATENCY_REPAIR_SPEC_V1_4_REVIEWER_NOTEBOOK`
**Revision:** `ra_medfilm_barr_phase34_efficiency_profiling_v1_4`

The v1.3 batch-1 latency implementation timed one 200-iteration block per CUDA-event pair and
mislabeled the five block totals as 1000 per-iteration observations. Throughput, memory, static
resource and analytical MAC evidence remain **valid and preserved**.


## 0. Setup

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path(r"D:\Softfors\jup")
AUD = ROOT / "reports" / "audits"
TAB = ROOT / "reports" / "tables"


def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


def tshow(name, n=None):
    df = pd.read_csv(TAB / name)
    print(name, "| shape:", df.shape)
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None, "display.width", 240):
        print(df.to_string(index=False))
    return df


print("Phase34 v1.4 latency-granularity + alias-deduplication REPAIR SPEC review")
print("READ-ONLY. No model construction, no forward, no timing, no protected data, no TEST.")


Phase34 v1.4 latency-granularity + alias-deduplication REPAIR SPEC review
READ-ONLY. No model construction, no forward, no timing, no protected data, no TEST.


## 1. Defect record

In [2]:
dr = jload(AUD / "phase34_v1_3_latency_granularity_and_alias_defect_record.json")
print("classification:", dr["classification"])
d1 = dr["DEFECT_1"]
print()
print("DEFECT 1:", d1["code"])
print("  recorded observations are:", d1["recorded_latency_observations_are"])
print("  block size:", d1["block_size"])
print("  reported unit semantics:", d1["reported_latency_unit_semantics"])
print("  NOT:", d1["not"])
print()
print("  observed evidence:", json.dumps(d1["observed_evidence"], indent=4))
d2 = dr["DEFECT_2"]
print()
print("DEFECT 2:", d2["code"])
print("  aliases:", d2["aliases"])
print("  deleted:", d2["deleted"], "| classification:", d2["classification"])


classification: POST_MEASUREMENT_LATENCY_IMPLEMENTATION_DEFECT__THROUGHPUT_MEMORY_STATIC_AND_ANALYTICAL_RESULTS_PRESERVED

DEFECT 1: LATENCY_BLOCK_ELAPSED_TIME_REPORTED_AS_PER_ITERATION_MILLISECONDS
  recorded observations are: 200_ITERATION_BLOCK_ELAPSED_DURATIONS
  block size: 200
  reported unit semantics: MILLISECONDS_PER_200_FORWARD_BLOCK
  NOT: MILLISECONDS_PER_FORWARD

  observed evidence: {
    "warmup": 50,
    "measured_blocks": 5,
    "iterations_per_block": 200,
    "iterations_total": 1000,
    "reported_GLOBAL_ONLY_C1_mean_ms": 3120.52431640625,
    "reported_SKIP_mean_ms": 3143.79365234375,
    "reported_EXECUTE_mean_ms": 3740.71328125,
    "latency_table_std_ms": "NaN for all six canonical rows",
    "percentile_anomaly": "p90 == p95 == p99 == max for all canonical rows"
}

DEFECT 2: ALIAS_EXECUTABLE_PATHS_WERE_MEASURED_SEPARATELY_DESPITE_DEDUPLICATION_REQUIREMENT
  aliases: {'G_COMPONENT': 'GLOBAL_ONLY_C1', 'FINAL_ROUTED_A8_C9': 'EXECUTE_CONTROLLED_PATH'}
  deleted: Fa

## 2. Source-level root cause

In [3]:
rc = jload(AUD / "phase34_v1_3_latency_source_root_cause.json")
print("source:", rc["source"], "| SHA:", rc["source_sha256"][:16], "...")
print("function:", rc["function"], "lines", rc["line_span"])
print("timing captures:", rc["timing_captures"])
print("not:", rc["not"])
print("conclusion:", rc["conclusion"])
print()
for e in rc["evidence_lines"]:
    print("  -", e)


source: scripts/analysis/phase34_efficiency_profiling_worker_v1_3.py | SHA: 295c64d94358aed7 ...
function: time_latency lines 278-311
timing captures: A. ONE_CUDA_EVENT_PAIR_PER_200_ITERATION_BLOCK
not: B. ONE_CUDA_EVENT_PAIR_PER_INDIVIDUAL_ITERATION
conclusion: CONFIRMED_FROM_SOURCE (not inferred from result magnitude)

  - L280-281: for k in range(C.WARMUP_ITERATIONS): fn(k % C.TASK_COUNT)  [warmup = 50]
  - L284: for _ in range(C.MEASUREMENT_BLOCKS):  [5 blocks]
  - L285-286: start = torch.cuda.Event(enable_timing=True); end = ...  [ONE event pair per BLOCK]
  - L287: _sync()
  - L289: start.record()   [before the 200-iteration loop]
  - L290-291: for k in range(C.MEASURED_ITERATIONS_PER_BLOCK): fn(k % C.TASK_COUNT)
  - L292: end.record()     [after the 200-iteration loop]
  - L293-294: _sync(); end.synchronize()
  - L295: per_block_ms.append(start.elapsed_time(end))  [5 block totals]
  - L297: ms = sorted(per_block_ms)  [5 values]
  - L298: n_observations = C.TOTAL_MEASURED_ITERATI

## 3. Validity partition

In [4]:
vp = jload(AUD / "phase34_v1_3_result_validity_partition.json")
for k, v in vp.items():
    if k in ("artifact", "phase", "section", "method", "revision", "generated_utc",
             "all_pass"):
        continue
    print(f"  {k:52s} = {v}")


  checkpoint_identity                                  = VALID
  parameter_counts                                     = VALID
  static_resource_profile                              = VALID
  analytical_MAC_profile                               = VALID
  batch8_throughput                                    = VALID
  GPU_memory_profile                                   = VALID
  protected_data_boundary                              = VALID
  batch1_reported_block_elapsed_values                 = VALID_AS_RAW_BLOCK_DURATION_EVIDENCE
  batch1_reported_values_as_per_forward_latency        = INVALID
  batch1_std                                           = INVALID_OR_UNAVAILABLE
  batch1_p50                                           = INVALID_AS_PER_ITERATION_PERCENTILE
  batch1_p90                                           = INVALID_AS_PER_ITERATION_PERCENTILE
  batch1_p95                                           = INVALID_AS_PER_ITERATION_PERCENTILE
  batch1_p99                             

## 4. ÷200 block-mean salvage arithmetic

In [5]:
sv = tshow("phase34_v1_3_block_mean_latency_salvage.csv")
print()
print("note: this is a MEAN-ONLY derivation from equal-size block totals.")
print("It cannot reconstruct per-iteration std / p50 / p90 / p95 / p99 / min / max.")


phase34_v1_3_block_mean_latency_salvage.csv | shape: (6, 5)
                    arm  reported_mean_block_elapsed_ms  block_size  block_average_mean_forward_ms                                    classification
         GLOBAL_ONLY_C1                     3120.524316         200                      15.602622 MEAN_ONLY_DERIVATION_FROM_EQUAL_SIZE_BLOCK_TOTALS
        R_PSI_COMPONENT                      937.517871         200                       4.687589 MEAN_ONLY_DERIVATION_FROM_EQUAL_SIZE_BLOCK_TOTALS
        B_PHI_COMPONENT                       76.909752         200                       0.384549 MEAN_ONLY_DERIVATION_FROM_EQUAL_SIZE_BLOCK_TOTALS
      S_OMEGA_COMPONENT                       45.097177         200                       0.225486 MEAN_ONLY_DERIVATION_FROM_EQUAL_SIZE_BLOCK_TOTALS
   SKIP_CONTROLLED_PATH                     3143.793652         200                      15.718968 MEAN_ONLY_DERIVATION_FROM_EQUAL_SIZE_BLOCK_TOTALS
EXECUTE_CONTROLLED_PATH                     37

## 5. Salvaged conditional mean runtime

In [6]:
tshow("phase34_v1_3_conditional_runtime_mean_salvage.csv")


phase34_v1_3_conditional_runtime_mean_salvage.csv | shape: (5, 4)
                    quantity     value     unit                                                                                                                                                                                                                        note
   L_SKIP_block_average_mean 15.718968       ms                                                                                                                                                                                             reported SKIP block total / 200
L_EXECUTE_block_average_mean 18.703566       ms                                                                                                                                                                                          reported EXECUTE block total / 200
                      p_SKIP  0.725714 fraction                                                                                   

,quantity,value,unit,note
0,L_SKIP_block_average_mean,15.718968,ms,reported SKIP block total / 200
1,L_EXECUTE_block_average_mean,18.703566,ms,reported EXECUTE block total / 200
2,p_SKIP,0.725714,fraction,635/875
3,p_EXECUTE,0.274286,fraction,240/875 (ETA 236 + ETR 4)
4,L_weighted_salvage,16.537601,ms,SALVAGED_MEAN_ONLY_WEIGHTED_HARDWARE_SPECIFIC_...


## 6. v1.4 six-arm latency registry

In [7]:
reg = tshow("phase34_latency_arm_registry_v1_4.csv")
print()
print("physical worker runs:", int(reg["physical_worker_runs"].sum()), "(expected 6)")


phase34_latency_arm_registry_v1_4.csv | shape: (8, 6)
                 arm_id  canonical                alias_of          execution_policy                                      components  physical_worker_runs
         GLOBAL_ONLY_C1       True                     NaN        EXECUTE_SEPARATELY                                               G                     1
        R_PSI_COMPONENT       True                     NaN        EXECUTE_SEPARATELY                                           R_psi                     1
        B_PHI_COMPONENT       True                     NaN        EXECUTE_SEPARATELY                                           B_phi                     1
      S_OMEGA_COMPONENT       True                     NaN        EXECUTE_SEPARATELY                                         S_omega                     1
   SKIP_CONTROLLED_PATH       True                     NaN        EXECUTE_SEPARATELY                                         G|B_phi                     1
EXECUTE_CONTROLL

## 7. v1.4 observation-unit contract

In [8]:
print("estimand            : PER_FORWARD_BATCH1_MODEL_COMPUTE_PATH_LATENCY_MS")
print("observation unit    : ONE model/path forward at batch size 1")
print("observations per arm: 1000 (5 blocks x 200)")
print("warmup              : 50 (same 0/1/2/3 task cycle)")
print("mean                : np.mean(values, dtype=float64)")
print("std                 : np.std(values, ddof=1, dtype=float64)")
print("percentiles         : np.quantile(values, [0.50,0.90,0.95,0.99], method='linear')")
print("min/max             : np.min / np.max")
print("no rounding, no outlier dropping, no winsorization, no retry on observed timing")
print()
print("raw observation artifact: 6 arms x 1000 = 6000 rows")


estimand            : PER_FORWARD_BATCH1_MODEL_COMPUTE_PATH_LATENCY_MS
observation unit    : ONE model/path forward at batch size 1
observations per arm: 1000 (5 blocks x 200)
warmup              : 50 (same 0/1/2/3 task cycle)
mean                : np.mean(values, dtype=float64)
std                 : np.std(values, ddof=1, dtype=float64)
percentiles         : np.quantile(values, [0.50,0.90,0.95,0.99], method='linear')
min/max             : np.min / np.max
no rounding, no outlier dropping, no winsorization, no retry on observed timing

raw observation artifact: 6 arms x 1000 = 6000 rows


## 8. v1.4 static preflight

In [9]:
pf = jload(AUD / "phase34_latency_preflight_v1_4.json")
print("inspection:", pf["method_of_inspection"])
print("checks:", pf["n_checks"], "| failed:", pf["n_failed"],
      "| pass:", pf["static_preflight_pass"])
print()
for k, v in pf["checks"].items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")


inspection: AST_PARSE_AND_SHA256_HASHING_ONLY
checks: 22 | failed: 0 | pass: True

  PASS  exact_six_arm_registry
  PASS  no_physical_alias_execution
  PASS  warmup_50
  PASS  five_blocks
  PASS  two_hundred_per_block
  PASS  one_thousand_total_observations
  PASS  one_cuda_event_interval_per_forward
  PASS  per_iteration_observations_persisted
  PASS  sample_std_ddof_1
  PASS  quantile_method_linear
  PASS  no_outlier_removal
  PASS  task_schedule_exact
  PASS  canonical_precision_exact
  PASS  fresh_worker_subprocess_per_arm
  PASS  throughput_not_rerun
  PASS  memory_not_rerun
  PASS  mac_profiler_not_run
  PASS  protected_data_paths_absent
  PASS  TEST_absent
  PASS  training_absent
  PASS  PHASE34_EXECUTE_gate_present
  PASS  preserved_v13_evidence_bound


## 9. v1.4 source freeze

In [10]:
sf = jload(AUD / "phase34_preexecution_source_freeze_v1_4.json")
for k, v in sf["frozen_sources"].items():
    print(f"  {v['sha256'][:16]}...  {k:30s} {v['path']}")
print()
print("v1.3 measured attempt preserved:", sf["v1_3_measured_attempt_preserved"])
print("v1.3 latency result not canonicalized:", sf["v1_3_latency_result_not_canonicalized"])
print("v1.3 throughput preserved valid:", sf["v1_3_throughput_preserved_valid"])
print("v1.3 memory preserved valid:", sf["v1_3_memory_preserved_valid"])
print("v1.3 static resources preserved valid:", sf["v1_3_static_resources_preserved_valid"])
print("v1.3 analytical MAC preserved valid:", sf["v1_3_analytical_MAC_preserved_valid"])


  369ae8c65b78f61a...  latency_worker_v1_4            scripts/analysis/phase34_latency_worker_v1_4.py
  1c96183c5554a3d3...  latency_harness_v1_4           scripts/analysis/phase34_latency_execution_v1_4.py
  f0fce5b58693d0dc...  latency_preflight_v1_4         scripts/analysis/phase34_latency_preflight_v1_4.py
  a8853417e1fd17ba...  repair_spec_generator_v1_4     scripts/analysis/phase34_latency_repair_spec_v1_4.py
  355f9fbf597c3d02...  repair_notebook_builder_v1_4   scripts/analysis/build_phase34_latency_repair_notebook_v1_4.py
  295c64d94358aed7...  worker_v1_3_frozen             scripts/analysis/phase34_efficiency_profiling_worker_v1_3.py
  ca74fe88a8314b60...  common_v1_2                    scripts/analysis/phase34_v1_2_common.py
  a04ff3e61b4d900f...  model_G                        src/models/film/global_medfilm.py
  4fe0abde3ec3a183...  model_R_psi                    src/models/barr/regional_refiner.py
  fc7c1bdf263ae455...  model_B_phi_authority          scripts/analysis/train_

## 10. Execution eligibility and no-execution audit

In [11]:
el = jload(AUD / "phase34_latency_execution_eligibility_v1_4.json")
print("latency rerun authorized:", el["latency_rerun_authorized"])
print("reason:", el["reason"])
print("throughput/memory/static/MAC rerun:",
      el["throughput_rerun_authorized"], el["memory_rerun_authorized"],
      el["static_resource_rerun_authorized"], el["analytical_MAC_rerun_authorized"])
print("alias physical rerun:", el["alias_physical_rerun_authorized"])
print()
ne = jload(AUD / "phase34_v1_4_no_execution_audit.json")
for k, v in ne["counters"].items():
    print(f"  {k:32s} = {v}")
print("all zero:", ne["all_zero"])
assert ne["all_zero"]


latency rerun authorized: True
reason: CORRECTION_OF_POST_MEASUREMENT_LATENCY_GRANULARITY_IMPLEMENTATION_DEFECT
throughput/memory/static/MAC rerun: False False False False
alias physical rerun: False

  model_constructions              = 0
  checkpoint_deserializations      = 0
  forwards                         = 0
  latency_warmups                  = 0
  latency_measured_iterations      = 0
  throughput_measurements          = 0
  memory_measurements              = 0
  mac_flop_profiler_executions     = 0
  router_holdout_raw_reads         = 0
  test_reads                       = 0
  training_steps                   = 0
all zero: True


## 11. Repair final check and terminal

In [12]:
fc = jload(AUD / "phase34_latency_repair_final_check_v1_4.json")
print("final check:", fc["n_checks"], "checks,", fc["n_failed"], "failed")
term = jload(AUD / "phase34_latency_repair_terminal_v1_4.json")
print()
print("terminal:")
print(" ", term["terminal"])


final check: 25 checks, 0 failed

terminal:
  PHASE34_LATENCY_REPAIR_SPEC_V1_4_COMPLETE__V1_3_200_ITERATION_BLOCK_TOTAL_LATENCY_MISLABELING_AND_ALIAS_DUPLICATE_EXECUTION_DISCLOSED__VALID_THROUGHPUT_MEMORY_STATIC_AND_ANALYTICAL_EVIDENCE_PRESERVED__SIX_DISTINCT_ARM_PER_FORWARD_CUDA_EVENT_LATENCY_PROTOCOL_WITH_6000_RAW_OBSERVATIONS_FROZEN__V1_4_SOURCE_FREEZE_COMPLETE__LATENCY_ONLY_REEXECUTION_ELIGIBLE_NOT_STARTED


---
## Summary

| item | value |
|---|---|
| v1.3 latency defect | 200-iteration block totals mislabeled as per-iteration ms |
| source proof | one CUDA-event pair per 200-iteration block (`time_latency` L285–L295) |
| v1.3 latency as per-forward | INVALID · std/p50/p90/p95/p99/min/max INVALID |
| v1.3 throughput / memory / static / MAC | **VALID and preserved** |
| mean-only salvage | reported block total ÷ 200 |
| v1.4 canonical arms | 6 (aliases not physically executed) |
| v1.4 observations | 1000 per-forward per arm · 6000 raw rows |
| v1.4 statistics | mean · std ddof=1 · linear quantiles · min/max |
| latency re-execution | ELIGIBLE, NOT started |
| Phase34 closure | NOT performed · Phase35 NOT started · TEST locked |
